In [1]:
from juliacall import Main

import holoviews as hv
hv.extension('bokeh', logo=False)
import panel as pn
pn.extension()
from panel.interact import interact

import numpy as np

In [2]:
%%julia
using LinearAlgebra, Polynomials, QuadGK

<div style="float:center;width:100%;text-align: center;"><strong style="height:60px;color:darkred;font-size:40px;">QR with Polynomials</strong></div>

**This is a simplified copy** of the following [**MIT 18.02 Notebook**](https://github.com/mitmath/1806/blob/master/notes/Orthogonal-Polynomials.ipynb)

# Inner Product

Consider the set of polynomials defined on $(-1, 1)$ with inner product

$\qquad
< f, g >\ = \int_{-1}^1{f(x) g(x) \ dx}, \quad \text{although, for improved readability, we might write }\quad f \cdot g\ = \int_{-1}^1{f(x) g(x) \ dx}
$

In [3]:
%%julia
polydot(p::Polynomial, q::Polynomial) = integrate(p*q, -1, 1);             # compute the dot product ⟨p,q⟩ = ∫p(x)q(x) on [-1,1]; This works for polys

polydot(f::Function,   q::Polynomial) = quadgk(x -> f(x)*q(x), -1,1, atol=1e-13, rtol=1e-11)[1]   # numerical integration
;

In [4]:
%%julia
p = Polynomial([ 0, 1, 0, 3, 0, 2])
q = Polynomial([ 1, 1])
println("p = $p")
println("q = $q")
println("<p, q> = ", polydot(1//1*p, q) );

p = x + 3*x^3 + 2*x^5
q = 1 + x
<p, q> = 256//105


# 1. Gram Schmidt Procedure

The Gram Schmidt Procedure works as before

$\qquad\begin{align}
w_1(x) =&\ v_1(x) \\
w_2(x) =&\ v_2(x) - \frac{<v_2,w_1>}{<w_1,w_1>} w_1 \\
w_3(x) =&\ v_3(x) - \frac{<v_3,w_1>}{<w_1,w_1>} w_1  - \frac{<v_3,w_2>}{<w_2,w_2>} w_2 \\
w_4(x) =&\ v_4(x) - \frac{<v_4,w_1>}{<w_1,w_1>} w_1  - \frac{<v_4,w_2>}{<w_2,w_2>} w_2   - \frac{<v_4,w_3>}{<w_3,w_3>} w_3 \\
\dots&
\end{align}$

#### **Example**

Use this inner product to run the Gram-Schmidt Procedure on $1, x, x^2, x^3, x^4, x^5, x^6$,<br>
normalize the polynomials such that 
$\displaystyle{q_i(x) = \frac{w_i(x)}{w_i(1)}}$

The resulting polynomials are known as the **Legendre Polynomials**

In [5]:
%%julia
function legendre_gramschmidt(n)
    legendre = [Polynomial([1//1])]
    for i = 1:n
        p = Polynomial([k == i ? 1//1 : 0//1 for k=0:i])
            p = p - sum( [ q * (polydot(q, p) // polydot(q,q)) for q in legendre] )
        push!(legendre, p / p(1))
    end
    return legendre
end

l6 = legendre_gramschmidt(6)
eval_lk(k,x) = l6[k].(x)
println.(l6);

1//1
x
-1//2 + 3//2*x^2
-3//2*x + 5//2*x^3
3//8 - 15//4*x^2 + 35//8*x^4
15//8*x - 35//4*x^3 + 63//8*x^5
-5//16 + 105//16*x^2 - 315//16*x^4 + 231//16*x^6


In [6]:
x = np.linspace(-1,1,300)
Main.eval_lk(3,x)
h=\
hv.Overlay( [hv.Curve((x,Main.eval_lk(i,x)), label=f"q_{i}").opts(muted_alpha=0) for i in range(1,len(Main.l6)+1) ] )*\
hv.VLine(0).opts(color="black", line_width=1)*\
hv.HLine(0).opts(color="black", line_width=1)

h.opts(width=700, legend_position="right", title="Legendre Polynomials")

:Overlay
   .Curve.Q_1 :Curve   [x]   (y)
   .Curve.Q_2 :Curve   [x]   (y)
   .Curve.Q_3 :Curve   [x]   (y)
   .Curve.Q_4 :Curve   [x]   (y)
   .Curve.Q_5 :Curve   [x]   (y)
   .Curve.Q_6 :Curve   [x]   (y)
   .Curve.Q_7 :Curve   [x]   (y)
   .VLine.I   :VLine   [x,y]
   .HLine.I   :HLine   [x,y]

In [7]:
hv.Overlay([hv.Curve((x,x**i),label=f"p_{i}").opts(muted_alpha=0) for i in range(7)]).opts(width=700, title="Powers of x", legend_position="right")

:Overlay
   .Curve.P_0 :Curve   [x]   (y)
   .Curve.P_1 :Curve   [x]   (y)
   .Curve.P_2 :Curve   [x]   (y)
   .Curve.P_3 :Curve   [x]   (y)
   .Curve.P_4 :Curve   [x]   (y)
   .Curve.P_5 :Curve   [x]   (y)
   .Curve.P_6 :Curve   [x]   (y)

# 2. Convert Power Series to Legendre Polynomials Expansion

The expansion of a function works as before. Let f(x) be a polynomial of degree $k$, and let $p_i(x), i=1,2,\dots k$ be the first $k$ Legendre Polynomials

$\qquad
f(x) = \sum_{i=0}^k{ \frac{ <f, p_i>}{<p_i,p_i>}}\ p_i(x)
$

In [8]:
%%julia
p = Polynomial([1,2,1,4])
println( "expand ", p, "\n")

coeffs = [polydot(p,q)/polydot(q,q) for q in l6]
for i in 1:length(coeffs)
    println( coeffs[i], "\t(", l6[i], ")" )
end

println( "\nCheck the expansion: ", sum( [polydot(p,q)/polydot(q,q) * q for q in l6] ) == p)

expand 1 + 2*x + x^2 + 4*x^3

4//3	(1//1)
22//5	(x)
2//3	(-1//2 + 3//2*x^2)
8//5	(-3//2*x + 5//2*x^3)
0//1	(3//8 - 15//4*x^2 + 35//8*x^4)
0//1	(15//8*x - 35//4*x^3 + 63//8*x^5)
0//1	(-5//16 + 105//16*x^2 - 315//16*x^4 + 231//16*x^6)

Check the expansion: true


# 3. Fit a Function

For more general functions, the expansion above yields a polynomial expansion that minimizes $\vert\vert\ f - p\ \vert\vert$, the least mean square distance between the function with respect to the chosen norm

In [9]:
%%julia
x      = range(-1,1,300)
fn(x)  = exp.(0.5x)                                     #  Fit this function (an exponential)

coeffs = [polydot(fn,p)/polydot(p,p) for p in l6]
terms  = [coeffs[i]*l6[i].(x) for i in 1:length(l6)]

function first_k(k)
    s = terms[1]
    for i=2:k
       s = s .+ terms[i]
    end
    s
end
println.(round.(coeffs, digits=5))
;

1.04219
0.51261
0.08483
0.00845
0.0006
3.0e-5
0.0


In [10]:
h = hv.Curve((Main.x, Main.fn(Main.x)), label="f(x)").opts(color="black", ylim=(0.58,1.7))
interact(lambda k: (h*hv.Curve( (Main.x, Main.first_k(k)),label="f_approx(x)" )).opts(width=500,legend_position="top_left"), k=(0,len(Main.l6), 1, 0))

Column(sizing_mode='fixed')
    [0] Column
        [0] IntSlider(end=7, name='k')
    [1] Row(sizing_mode='fixed')
        [0] HoloViews(Overlay, height=300, name='interactive00606', sizing_mode='fixed', width=500)

In [11]:
%%julia
t(x) = 1 - abs(x)
t_exact = t.(x)

L16 = legendre_gramschmidt(16)

function approx(n)
    sum([polydot(t,p)/polydot(p,p) for p in L16[1:n]] .* L16[1:n]).(x)
end
#approx(16)
;

In [12]:
h = hv.Curve((Main.x, Main.t_exact), label="f(x)").opts(color="black", ylim=(-0.4,1.2))
interact(lambda k: (h*hv.Curve( (Main.x, Main.approx(k)),label="f_approx(x)" )).opts(width=500,legend_position="top"), k=(1,len(Main.L16)-1, 2, 0))

Column(sizing_mode='fixed')
    [0] Column
        [0] IntSlider(end=16, name='k', start=1, step=2, value=1)
    [1] Row(sizing_mode='fixed')
        [0] HoloViews(Overlay, height=300, name='interactive00671', sizing_mode='fixed', width=500)